In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from scipy.stats import norm

# Let me first import the data of stock 
Sec_Data = yf.download("RELIANCE.NS",start='2000-1-1',end='2026-5-22',auto_adjust=False)['Adj Close']

#let's name the variables
#S is Current Stock Price
S = Sec_Data.iloc[-1]

#T is Time duration in years
T = 1

#Rf is Risk Free Return
Rf = 0.07

#K Stike Price
K = 1600

#Std is Normal Standard Deviation (To Annualised Std I have sqrt the time not product the time as we do with variance)
Log_Return = np.log(1+Sec_Data.pct_change())
Std = Log_Return.std()*np.sqrt(250)

#Lets work on the 1 parts of formula D1 which defines the Z-score
def D1(S,T,Rf,K,Std):
    Numerator1 = np.log(S/K)+(Rf+Std**2/2)*T
    Denominator1 = Std * np.sqrt(T)
    return Numerator1 / Denominator1

def D2(S,T,Rf,K,Std):
    Numerator2 = np.log(S/K)+(Rf-Std**2/2)*T
    Denominator2 = Std * np.sqrt(T)
    return Numerator2 / Denominator2

def BSM_Call_Price(S,T,Rf,K,Std):
    d_1 = D1(S,T,Rf,K,Std) #Calculate the Z-Score Boundry line for D1 and D2
    d_2 = D2(S,T,Rf,K,Std)
    Stock_filter = norm.cdf(d_1) #the Delta (My Share of the upside)
    Cash_filter = norm.cdf(d_2) #Probability I will exercise and pay K
    Discount = K * np.exp(-Rf*T) #Discounting Factor
    Call_Price = (S*Stock_filter)-(Cash_filter*Discount)
    return {"Price":Call_Price,"d1":d_1,"d2":d_2,"Cash":Cash_filter,"Stock":Stock_filter}

# Assign the name for output
bsm_output = BSM_Call_Price(S, T, Rf, K, Std)

# FIX: Use .iloc[0] instead of [0] becuase I am getting an warning stating in future it will be changed
d2_num = bsm_output["d2"].iloc[0]
d1_num = bsm_output["d1"].iloc[0]
price_num = bsm_output["Price"].iloc[0]
Cash_fil = bsm_output["Cash"][0]
Stock_fil = bsm_output["Stock"][0]
S0 = S.iloc[0]

print(f"--- Internal BSM Breakdown ---")
print(f"d2 (Distance to Strike): {d2_num:.4f} standard deviations")
print(f"d1 (Leverage-Adjusted Distance): {d1_num:.4f} standard deviations")
print(f"Cash Filter (Probability of paying K): {Cash_fil * 100:.2f}%") #It gives me the % of chance that this option will expire "in-the-money
print(f"Stock Filter (Delta / Upside Share): {Stock_fil * 100:.2f}%") #It gives the worth of % ownership in the 1 whole share
print(f"Call Price of Option is: ₹{price_num:.2f} when Share price is ₹{S0:.2f}")

print(f"\n D2 is about Probability: What are the clean, unweighted odds that I finish above the strike? {Cash_fil*100:.2f}%")
print(f"\n D1 is about Value: If I do finish above the strike,\
how deep into the money am I likely to be, and how much exposure do I have? {Stock_fil*100:.2f}%")

[*********************100%***********************]  1 of 1 completed

--- Internal BSM Breakdown ---
d2 (Distance to Strike): -0.4526 standard deviations
d1 (Leverage-Adjusted Distance): 0.0670 standard deviations
Cash Filter (Probability of paying K): 32.54%
Stock Filter (Delta / Upside Share): 52.67%
Call Price of Option is: ₹225.41 when Share price is ₹1349.60

 D2 is about Probability: What are the clean, unweighted odds that I finish above the strike? 32.54%

 D1 is about Value: If I do finish above the strike,how deep into the money am I likely to be, and how much exposure do I have? 52.67%
